In [6]:


from pathlib import Path

files = sorted(Path("../Data/Raw").glob("MD_Drug_Utilization_Data 20*.csv"))

cols = [
    "Number of Prescriptions",
    "Units Reimbursed",
    "Total Amount Reimbursed",
    "Medicaid Amount Reimbursed"
]

results = []

for file in files:
    for chunk in pd.read_csv(file, chunksize=50000, low_memory=False):
        md = chunk.loc[chunk["State"] == "MD"].copy()
        md[cols] = md[cols].apply(pd.to_numeric, errors="coerce")

        results.append(
            md.groupby(["Year", "Quarter", "Utilization Type"])[cols]
            .sum(min_count=1)
        )

quarterly = (
    pd.concat(results)
    .groupby(level=[0, 1, 2])
    .sum(min_count=1)
    .reset_index()
    .sort_values(["Year", "Quarter", "Utilization Type"])
)

quarterly.to_csv(
    "../Data/Processed/pharmacy_quarterly_clean.csv",
    index=False
)

print("Files found:", len(files))
print("Quarterly rows:", len(quarterly))
display(quarterly)



Files found: 4
Quarterly rows: 26


,Year,Quarter,Utilization Type,Number of Prescriptions,Units Reimbursed,Total Amount Reimbursed,Medicaid Amount Reimbursed
0,2023,1,FFSU,1334524.0,7.103016e+07,1.465016e+08,1.439837e+08
1,2023,1,MCOU,2680899.0,1.465973e+08,3.063420e+08,3.016693e+08
2,2023,2,FFSU,1339539.0,7.066574e+07,1.454232e+08,1.431884e+08
3,2023,2,MCOU,2712506.0,1.477660e+08,3.082800e+08,3.034741e+08
4,2023,3,FFSU,1308643.0,6.808378e+07,1.417192e+08,1.397771e+08
5,2023,3,MCOU,2498839.0,1.347033e+08,2.866635e+08,2.823320e+08
6,2023,4,FFSU,1305711.0,6.736084e+07,1.409227e+08,1.390763e+08
7,2023,4,MCOU,2595276.0,1.438478e+08,2.754952e+08,2.715736e+08
8,2024,1,FFSU,1317167.0,7.231161e+07,1.545885e+08,1.525052e+08
9,2024,1,MCOU,2566593.0,1.445701e+08,2.826108e+08,2.782763e+08


In [7]:

import sqlite3

db = "../Database/maryland_medicaid.db"

with sqlite3.connect(db) as conn:
    quarterly.to_sql(
        "pharmacy_quarterly",
        conn,
        if_exists="replace",
        index=False
    )

    pharmacy_total = (
        quarterly.groupby(["Year", "Quarter"], as_index=False)
        .agg({
            "Number of Prescriptions": "sum",
            "Units Reimbursed": "sum",
            "Total Amount Reimbursed": "sum",
            "Medicaid Amount Reimbursed": "sum"
        })
    )

    pharmacy_total.to_sql(
        "pharmacy_quarterly_total",
        conn,
        if_exists="replace",
        index=False
    )

print("Pharmacy tables saved!")


Pharmacy tables saved!


In [8]:

with sqlite3.connect(db) as conn:
    tables = pd.read_sql(
        "SELECT name FROM sqlite_master WHERE type='table'",
        conn
    )
    print(tables)

    check = pd.read_sql(
        'SELECT * FROM pharmacy_quarterly_total ORDER BY Year, Quarter',
        conn
    )
    display(check)


                        name
0                 enrollment
1                 acute_care
2     eligibility_processing
3              current_trend
4           current_analysis
5     historical_utilization
6            powerbi_current
7         powerbi_historical
8      powerbi_largest_drops
9         pharmacy_quarterly
10  pharmacy_quarterly_total


,Year,Quarter,Number of Prescriptions,Units Reimbursed,Total Amount Reimbursed,Medicaid Amount Reimbursed
0,2023,1,4015423.0,2.176275e+08,4.528436e+08,4.456530e+08
1,2023,2,4052045.0,2.184317e+08,4.537032e+08,4.466625e+08
2,2023,3,3807482.0,2.027871e+08,4.283827e+08,4.221090e+08
3,2023,4,3900987.0,2.112087e+08,4.164179e+08,4.106499e+08
4,2024,1,3883760.0,2.168817e+08,4.371993e+08,4.307815e+08
5,2024,2,3871597.0,2.164346e+08,4.474481e+08,4.409558e+08
6,2024,3,3680311.0,1.983246e+08,4.388244e+08,4.324314e+08
7,2024,4,3793659.0,2.047046e+08,4.233713e+08,4.176424e+08
8,2025,1,3678413.0,2.013703e+08,4.229426e+08,4.167564e+08
9,2025,2,3687196.0,2.004371e+08,4.314412e+08,4.249211e+08


In [12]:

query = """
WITH enrollment_q AS (
    SELECT
        CAST(strftime('%Y', Month) AS INTEGER) AS Year,
        CAST((CAST(strftime('%m', Month) AS INTEGER) + 2) / 3 AS INTEGER) AS Quarter,
        AVG("Total Medicaid Enrollment") AS avg_enrollment,
        COUNT(DISTINCT Month) AS enrollment_months
    FROM enrollment
    GROUP BY 1, 2
),
eligibility_q AS (
    SELECT
        CAST(strftime('%Y', Month) AS INTEGER) AS Year,
        CAST((CAST(strftime('%m', Month) AS INTEGER) + 2) / 3 AS INTEGER) AS Quarter,
        SUM("Beneficiaries Disenrolled at Renewal (Total)") AS disenrollments,
        SUM("Beneficiaries Disenrolled for Procedural Reasons at Renewal") AS procedural_disenrollments,
        COUNT(DISTINCT Month) AS eligibility_months
    FROM eligibility_processing
    GROUP BY 1, 2
)
SELECT
    p.*,
    e.avg_enrollment,
    d.disenrollments,
    d.procedural_disenrollments,
    e.enrollment_months,
    d.eligibility_months,
    CASE
        WHEN e.avg_enrollment > 0 AND e.enrollment_months = 3
        THEN p."Number of Prescriptions" * 1000.0 / e.avg_enrollment
    END AS prescriptions_per_1000
FROM pharmacy_quarterly_total p
LEFT JOIN enrollment_q e
    ON p.Year = e.Year AND p.Quarter = e.Quarter
LEFT JOIN eligibility_q d
    ON p.Year = d.Year AND p.Quarter = d.Quarter
ORDER BY p.Year, p.Quarter
"""

with sqlite3.connect(db) as conn:
    analysis = pd.read_sql(query, conn)
    analysis.to_sql(
        "pharmacy_enrollment_analysis",
        conn,
        if_exists="replace",
        index=False
    )

display(analysis)




,Year,Quarter,Number of Prescriptions,Units Reimbursed,Total Amount Reimbursed,Medicaid Amount Reimbursed,avg_enrollment,disenrollments,procedural_disenrollments,enrollment_months,eligibility_months,prescriptions_per_1000
0,2023,1,4015423.0,2.176275e+08,4.528436e+08,4.456530e+08,1.535019e+06,NaN,NaN,3,NaN,2615.878370
1,2023,2,4052045.0,2.184317e+08,4.537032e+08,4.466625e+08,1.548545e+06,64930.0,41718.0,3,2.0,2616.678882
2,2023,3,3807482.0,2.027871e+08,4.283827e+08,4.221090e+08,1.521006e+06,65376.0,23621.0,3,3.0,2503.265053
3,2023,4,3900987.0,2.112087e+08,4.164179e+08,4.106499e+08,1.508194e+06,105186.0,85591.0,3,3.0,2586.528084
4,2024,1,3883760.0,2.168817e+08,4.371993e+08,4.307815e+08,1.453079e+06,89704.0,79608.0,3,3.0,2672.780276
5,2024,2,3871597.0,2.164346e+08,4.474481e+08,4.409558e+08,1.427548e+06,100286.0,88396.0,3,3.0,2712.060187
6,2024,3,3680311.0,1.983246e+08,4.388244e+08,4.324314e+08,1.369208e+06,128608.0,108088.0,3,3.0,2687.912937
7,2024,4,3793659.0,2.047046e+08,4.233713e+08,4.176424e+08,1.315298e+06,130821.0,112788.0,3,3.0,2884.258168
8,2025,1,3678413.0,2.013703e+08,4.229426e+08,4.167564e+08,1.261696e+06,109096.0,97020.0,3,3.0,2915.450337
9,2025,2,3687196.0,2.004371e+08,4.314412e+08,4.249211e+08,1.232110e+06,82756.0,71885.0,3,3.0,2992.587510


In [13]:

display(analysis[[
    "Year",
    "Quarter",
    "avg_enrollment",
    "enrollment_months",
    "disenrollments",
    "eligibility_months",
    "Number of Prescriptions",
    "prescriptions_per_1000"
]])


,Year,Quarter,avg_enrollment,enrollment_months,disenrollments,eligibility_months,Number of Prescriptions,prescriptions_per_1000
0,2023,1,1.535019e+06,3,NaN,NaN,4015423.0,2615.878370
1,2023,2,1.548545e+06,3,64930.0,2.0,4052045.0,2616.678882
2,2023,3,1.521006e+06,3,65376.0,3.0,3807482.0,2503.265053
3,2023,4,1.508194e+06,3,105186.0,3.0,3900987.0,2586.528084
4,2024,1,1.453079e+06,3,89704.0,3.0,3883760.0,2672.780276
5,2024,2,1.427548e+06,3,100286.0,3.0,3871597.0,2712.060187
6,2024,3,1.369208e+06,3,128608.0,3.0,3680311.0,2687.912937
7,2024,4,1.315298e+06,3,130821.0,3.0,3793659.0,2884.258168
8,2025,1,1.261696e+06,3,109096.0,3.0,3678413.0,2915.450337
9,2025,2,1.232110e+06,3,82756.0,3.0,3687196.0,2992.587510


In [14]:

analysis = analysis.sort_values(["Year", "Quarter"]).copy()

analysis["enrollment_change_pct"] = (
    analysis["avg_enrollment"].pct_change() * 100
)

analysis["prescription_change_pct"] = (
    analysis["Number of Prescriptions"].pct_change() * 100
)

analysis["utilization_change_pct"] = (
    analysis["prescriptions_per_1000"].pct_change() * 100
)

# Exclude incomplete eligibility quarters
analysis.loc[
    analysis["eligibility_months"] != 3,
    ["disenrollments", "procedural_disenrollments"]
] = float("nan")

display(analysis[[
    "Year", "Quarter",
    "enrollment_change_pct",
    "prescription_change_pct",
    "utilization_change_pct",
    "disenrollments"
]].round(2))


,Year,Quarter,enrollment_change_pct,prescription_change_pct,utilization_change_pct,disenrollments
0,2023,1,NaN,NaN,NaN,NaN
1,2023,2,0.88,0.91,0.03,NaN
2,2023,3,-1.78,-6.04,-4.33,65376.0
3,2023,4,-0.84,2.46,3.33,105186.0
4,2024,1,-3.65,-0.44,3.33,89704.0
5,2024,2,-1.76,-0.31,1.47,100286.0
6,2024,3,-4.09,-4.94,-0.89,128608.0
7,2024,4,-3.94,3.08,7.30,130821.0
8,2025,1,-4.08,-3.04,1.08,109096.0
9,2025,2,-2.34,0.24,2.65,82756.0


In [15]:

variables = [
    "enrollment_change_pct",
    "prescription_change_pct",
    "utilization_change_pct",
    "disenrollments",
    "procedural_disenrollments"
]

display(analysis[variables].corr().round(3))


,enrollment_change_pct,prescription_change_pct,utilization_change_pct,disenrollments,procedural_disenrollments
enrollment_change_pct,1.000,0.125,-0.324,-0.659,-0.579
prescription_change_pct,0.125,1.000,0.898,0.378,0.444
utilization_change_pct,-0.324,0.898,1.000,0.599,0.633
disenrollments,-0.659,0.378,0.599,1.000,0.928
procedural_disenrollments,-0.579,0.444,0.633,0.928,1.000


In [16]:

analysis["enrollment_lag1"] = (
    analysis["enrollment_change_pct"].shift(1)
)

analysis["disenrollment_lag1"] = (
    analysis["disenrollments"].shift(1)
)

lag_results = analysis[[
    "enrollment_lag1",
    "disenrollment_lag1",
    "prescription_change_pct",
    "utilization_change_pct"
]].corr().round(3)

display(lag_results)


,enrollment_lag1,disenrollment_lag1,prescription_change_pct,utilization_change_pct
enrollment_lag1,1.000,-0.598,-0.398,-0.519
disenrollment_lag1,-0.598,1.000,-0.111,0.242
prescription_change_pct,-0.398,-0.111,1.000,0.898
utilization_change_pct,-0.519,0.242,0.898,1.000


In [17]:

analysis = analysis.sort_values(["Year", "Quarter"]).copy()

for col in ["avg_enrollment", "Number of Prescriptions", "prescriptions_per_1000"]:
    analysis[col + "_yoy_pct"] = (
        analysis[col].pct_change(4) * 100
    )

display(analysis[[
    "Year",
    "Quarter",
    "avg_enrollment_yoy_pct",
    "Number of Prescriptions_yoy_pct",
    "prescriptions_per_1000_yoy_pct"
]].round(2))


,Year,Quarter,avg_enrollment_yoy_pct,Number of Prescriptions_yoy_pct,prescriptions_per_1000_yoy_pct
0,2023,1,NaN,NaN,NaN
1,2023,2,NaN,NaN,NaN
2,2023,3,NaN,NaN,NaN
3,2023,4,NaN,NaN,NaN
4,2024,1,-5.34,-3.28,2.18
5,2024,2,-7.81,-4.45,3.65
6,2024,3,-9.98,-3.34,7.38
7,2024,4,-12.79,-2.75,11.51
8,2025,1,-13.17,-5.29,9.08
9,2025,2,-13.69,-4.76,10.34


In [18]:

with sqlite3.connect(db) as conn:
    analysis.to_sql(
        "pharmacy_enrollment_analysis",
        conn,
        if_exists="replace",
        index=False
    )

analysis.to_csv(
    "../Data/Processed/pharmacy_enrollment_analysis.csv",
    index=False
)

print("Final pharmacy analysis saved!")


Final pharmacy analysis saved!
